# Node embeddings (DeepWalk, node2vec) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sigmoid(x): return 1/(1+np.exp(-x))
def softmax(x):
    x=np.asarray(x,dtype=float); e=np.exp(x-x.max()); return e/e.sum()
def draw_graph(A, pos=None, values=None, title=''):
    A=np.asarray(A); n=A.shape[0]
    if pos is None:
        ang=np.linspace(0,2*np.pi,n,endpoint=False); pos=np.c_[np.cos(ang),np.sin(ang)]
    plt.figure(figsize=(4,4))
    for i in range(n):
        for j in range(i+1,n):
            if A[i,j]!=0: plt.plot([pos[i,0],pos[j,0]],[pos[i,1],pos[j,1]],c='0.75',lw=1+2*abs(A[i,j]))
    c=values if values is not None else np.arange(n)
    plt.scatter(pos[:,0],pos[:,1],c=c,s=320,cmap='viridis',edgecolor='k',zorder=3)
    for i,(x,y) in enumerate(pos): plt.text(x,y,str(i),ha='center',va='center',color='white',weight='bold')
    plt.axis('off'); plt.title(title)
print('setup ok')

## Random walks make graph neighborhoods look like sentences
DeepWalk and node2vec learn vectors by making nearby walk-context nodes close. The examples use transition matrices, walk co-occurrences, a tiny SVD embedding, and the node2vec return/in-out bias.

In [ ]:
# 1) one-step random-walk transition probabilities
A=np.array([[0,1,0,0],[1,0,1,0],[0,1,0,1],[0,0,1,0]],float); P=A/A.sum(1,keepdims=True)
plt.figure(figsize=(4,3)); plt.imshow(P,cmap='Blues'); plt.colorbar(); plt.title('transition P')
assert np.allclose(P[1],[0.5,0,0.5,0])

In [ ]:
# 2) two-step probabilities reveal broader context
A=np.array([[0,1,0,0],[1,0,1,0],[0,1,0,1],[0,0,1,0]],float); P=A/A.sum(1,keepdims=True); P2=P@P
plt.figure(figsize=(4,3)); plt.imshow(P2,cmap='Purples'); plt.colorbar(); plt.title('two-step context P^2')
assert abs(P2[0,2]-0.5)<1e-9

In [ ]:
# 3) fixed walks create a co-occurrence matrix
walks=[[0,1,2,3],[3,2,1,0],[0,1,0,1]]; C=np.zeros((4,4))
for w in walks:
    for i,u in enumerate(w):
        for j in range(max(0,i-1),min(len(w),i+2)):
            if i!=j: C[u,w[j]]+=1
plt.figure(figsize=(4,3)); plt.imshow(C,cmap='magma'); plt.colorbar(); plt.title('window-1 co-occurrence')
assert C[1,0]==5 and C[1,2]==2

In [ ]:
# 4) SVD of co-occurrence gives a tiny embedding
C=np.array([[0,5,0,0],[5,0,2,0],[0,2,0,2],[0,0,2,0]],float); U,S,Vt=np.linalg.svd(C); Z=U[:,:2]*np.sqrt(S[:2])
plt.figure(figsize=(4,4)); plt.scatter(Z[:,0],Z[:,1],s=250); [plt.text(Z[i,0],Z[i,1],str(i),ha='center',va='center') for i in range(4)]; plt.title('2D node embedding')
assert np.linalg.norm(Z[0]-Z[2])<np.linalg.norm(Z[0]-Z[3])

In [ ]:
# 5) node2vec bias: return probability grows when p is small
def n2v_probs(prev,cur,neighbors,p,q):
    weights=[]
    for x in neighbors:
        weights.append(1/p if x==prev else 1 if abs(x-prev)==1 else 1/q)
    weights=np.array(weights,float); return weights/weights.sum()
pr=n2v_probs(0,1,[0,2],p=0.5,q=2)
plt.figure(figsize=(5,3)); plt.bar(['return 0','outward 2'],pr); plt.title('node2vec biased transition')
assert np.allclose(pr,[0.8,0.2])